# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathamTumminakatti/ML1/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1: The Anatomy of Growing Content

The paper reports that growing content was longer and younger than declining content. Growing pages averaged approximately 3,180 words and 184 days of age, while declining pages averaged 2,311 words and 230 days of age.

### Methodology Question

The label comes from the observed trend direction: pages were grouped as rising or falling based on their impression movement. This is an observational comparison, not an independently controlled experiment.

The comparison supports a directional association between content age, content depth, and growth status. However, it does not prove that increasing word count or reducing content age directly causes growth.

Other factors such as topic, search intent, client characteristics, existing visibility, and content quality may influence both the content structure and the observed trend. A stronger validation design would control for these factors or compare similar pages over time.

Therefore, I interpret this finding as useful decision-support evidence rather than a causal rule that longer content automatically performs better.

## Finding 2: The Content Performance Curve

The paper reports that content performance was strongest around the 61–90 day age band and declined sharply around the 271–365 day range. It also reports a recovery among some 365+ day pages, especially pages that had been refreshed.

### Methodology Question

The label here is the observed health-score level grouped by content-age tier. The finding is based on cross-sectional comparisons between age groups rather than a controlled longitudinal experiment following the same pages throughout their lifecycle.

The age pattern supports a directional lifecycle association, but it does not establish that age alone causes performance decline. Older pages may differ from newer pages in topic, authority, publishing quality, historical performance, and likelihood of receiving updates.

The reported recovery among older pages should also be interpreted carefully because refreshed pages may be systematically different from pages that were not refreshed. The paper itself notes that the result should not be read as evidence that age naturally reverses decline.

A stronger validation design would track the same pages over time and compare refreshed and unrefreshed pages with similar prior performance and content characteristics.

Therefore, this finding is useful for prioritizing review windows, but it should not be treated as proof that every page declines at the same age or that refreshing always causes recovery.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Validation Comparison

I first evaluate the Logistic Regression model using a standard random train/test split. I then compare it with a grouped split based on `client_id`.

The random split may place content from the same client in both training and test sets. This can make the task easier because client-specific patterns may appear in both sets.

The grouped split keeps each client entirely in either the training or test set. This better tests whether the model can generalize to clients it has not seen before.

The comparison uses the same target, feature set, preprocessing, model, and evaluation metrics. The difference between the two results is treated as a validation finding rather than automatically as model improvement.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load data
data = pd.read_csv("/content_refresh_anonymized.csv")

# Recreate the Week-4 baseline score
baseline = data.copy()

baseline["age_score"] = (
    baseline["content_age_days"] / baseline["content_age_days"].max()
).clip(0, 1)

baseline["stale_score"] = (
    baseline["days_since_last_update"] / baseline["days_since_last_update"].max()
).clip(0, 1)

baseline["impression_score"] = (
    baseline["impressions_90d"] / baseline["impressions_90d"].max()
).clip(0, 1)

baseline["ctr_opportunity"] = (
    1 - baseline["ctr"] / baseline["ctr"].max()
).clip(0, 1)

baseline["baseline_score"] = (
    0.35 * baseline["age_score"]
    + 0.30 * baseline["stale_score"]
    + 0.25 * baseline["impression_score"]
    + 0.10 * baseline["ctr_opportunity"]
)

# Define target using the training-independent baseline threshold
threshold = baseline["baseline_score"].quantile(0.80)
data["refresh_target"] = (
    baseline["baseline_score"] >= threshold
).astype(int)

# Same features used in ML-08
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update"
]

categorical_features = [
    "competition_level", "content_type", "main_intent", "provider_used",
    "model_used", "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier"
]

feature_columns = numeric_features + categorical_features

X = data[feature_columns]
y = data["refresh_target"]
groups = data["client_id"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

def build_model():
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0)
    }

# --------------------------------------------------
# BEFORE: Random train/test split
# --------------------------------------------------
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = build_model()

random_metrics = evaluate_model(
    random_model,
    X_train_random,
    X_test_random,
    y_train_random,
    y_test_random
)

# --------------------------------------------------
# AFTER: Grouped split by client_id
# --------------------------------------------------
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]
y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

grouped_model = build_model()

grouped_metrics = evaluate_model(
    grouped_model,
    X_train_grouped,
    X_test_grouped,
    y_train_grouped,
    y_test_grouped
)

# --------------------------------------------------
# Display comparison
# --------------------------------------------------
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Random Split (Before)": [
        random_metrics["Accuracy"],
        random_metrics["Precision"],
        random_metrics["Recall"],
        random_metrics["F1"]
    ],
    "Grouped Split (After)": [
        grouped_metrics["Accuracy"],
        grouped_metrics["Precision"],
        grouped_metrics["Recall"],
        grouped_metrics["F1"]
    ]
})

display(comparison.round(4))

print("Overall positive-class base rate:", round(y.mean(), 4))
print("Random split train rows:", len(X_train_random))
print("Random split test rows:", len(X_test_random))
print("Grouped split train rows:", len(X_train_grouped))
print("Grouped split test rows:", len(X_test_grouped))
print("Grouped train clients:", data.iloc[train_idx]["client_id"].nunique())
print("Grouped test clients:", data.iloc[test_idx]["client_id"].nunique())
print(
    "Client overlap:",
    len(
        set(data.iloc[train_idx]["client_id"])
        & set(data.iloc[test_idx]["client_id"])
    )
)

,Metric,Random Split (Before),Grouped Split (After)
0,Accuracy,0.9923,0.9304
1,Precision,0.9906,0.9549
2,Recall,0.9708,0.8129
3,F1,0.9806,0.8782


Overall positive-class base rate: 0.2
Random split train rows: 24000
Random split test rows: 6000
Grouped split train rows: 23837
Grouped split test rows: 6163
Grouped train clients: 25
Grouped test clients: 7
Client overlap: 0


## Before/After Interpretation

The random split produced an F1 score of 0.9806, while the grouped client-level split produced an F1 score of 0.8782.

Accuracy also decreased from 0.9923 to 0.9304. The largest change was in recall, which decreased from 0.9708 to 0.8129. This means the model missed more refresh-priority items when evaluated on clients that were not present in training.

The grouped split is a more honest validation design for this dataset because all rows belonging to a client are kept in the same partition. The model therefore has to generalize to unseen client patterns instead of benefiting from similar content from the same client appearing in both sets.

The positive-class base rate was 0.20, meaning 20% of records were labeled as refresh-priority. Therefore, accuracy alone is not sufficient to judge performance. The grouped F1 score of 0.8782 and recall of 0.8129 provide a more useful view of the model's limitations.

The gap between the random and grouped results is itself an important finding. It suggests that the random split gave an optimistic estimate of generalization performance.

The grouped result should be treated as the more credible estimate for this experiment, although it still measures agreement with a rule-derived target rather than actual real-world refresh success.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

# 3. Leakage Audit

The target represents refresh priority based on the Week-4 baseline score.

I checked the final feature set for three major leakage risks:

1. Label-derived features: columns directly used to construct the target or closely related fields.
2. Future or overlapping windows: metrics that may contain information from the same period as the target.
3. Decision-derived features: existing scores or workflow labels that reproduce the previous decision rule.

The model does not use `trend_pct` or `trend_direction`, because these are directly related to content performance change and could reveal information about the target.

The identifiers `content_id` and `client_id` are also excluded from model features. `client_id` is used only for grouped splitting.

The baseline score itself is used to define the target and is not included as a feature. This avoids simply teaching the model to reproduce the existing score.

Missing values are handled through the preprocessing pipeline using median imputation for numeric columns and most-frequent imputation plus one-hot encoding for categorical columns.

In [3]:
# Leakage audit for the final feature set

suspect_columns = [
    "trend_pct",
    "trend_direction",
    "baseline_score",
    "refresh_target",
    "content_id",
    "client_id"
]

print("LEAKAGE AUDIT")
print("=" * 50)

print("\nSuspect columns checked:")
for column in suspect_columns:
    print(f"- {column}: {'present in data' if column in data.columns else 'not present'}")

print("\nFinal model feature count:", len(feature_columns))

print("\nSuspect columns included in model features:")
suspect_in_features = [
    column for column in suspect_columns if column in feature_columns
]

if suspect_in_features:
    print(suspect_in_features)
else:
    print("None")

print("\nTrend-related features in final feature set:")
trend_features = [
    column for column in feature_columns
    if "trend" in column.lower()
]

print(trend_features if trend_features else "None")

print("\nIdentifier columns in final feature set:")
identifier_features = [
    column for column in feature_columns
    if column in ["content_id", "client_id"]
]

print(identifier_features if identifier_features else "None")

print("\nMissing values in final feature columns:")
missing_summary = data[feature_columns].isna().sum()
print(missing_summary[missing_summary > 0].sort_values(ascending=False).head(15))

print("\nLeakage audit conclusion:")
if not suspect_in_features and not trend_features and not identifier_features:
    print("No explicitly identified label-derived, trend-derived, or identifier columns are included.")
else:
    print("Review the listed columns before trusting the model.")

LEAKAGE AUDIT

Suspect columns checked:
- trend_pct: present in data
- trend_direction: present in data
- baseline_score: not present
- refresh_target: present in data
- content_id: present in data
- client_id: present in data

Final model feature count: 34

Suspect columns included in model features:
None

Trend-related features in final feature set:
None

Identifier columns in final feature set:
None

Missing values in final feature columns:
provider_used        21438
word_count            7699
word_count_tier       7699
char_count_tier       7699
char_count            7699
model_used            5733
competition_level     2610
cpc                   2468
competition           2468
search_volume         2468
main_intent           2374
dtype: int64

Leakage audit conclusion:
No explicitly identified label-derived, trend-derived, or identifier columns are included.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# 4. Claim Rewrite

## Original Claim

The Logistic Regression model achieves approximately 99% accuracy and can reliably identify content that should be refreshed.

## Evidence-Based Rewrite

Under a random train/test split, the Logistic Regression model achieved an F1 score of 0.9806 and accuracy of 0.9923.

However, under a grouped client-level split, performance decreased to an F1 score of 0.8782, accuracy of 0.9304, and recall of 0.8129.

Therefore, the grouped result is a more credible estimate of generalization to unseen clients. The model shows useful agreement with the rule-derived refresh-priority target, but its performance should not be described as guaranteed real-world refresh success.

The model should be presented as decision-support evidence for prioritizing content review. Further validation using independently observed outcomes after content refresh would be required to determine whether it improves actual refresh performance.

## Safe Claim Language

I will use terms such as:

- Observed
- Measured
- Directional
- Associated with
- Decision-support
- More credible validation estimate
- Rule-derived target

I will avoid claims such as:

- Guarantees improvement
- Causes traffic growth
- Reliably predicts real-world success
- Proves that refreshing content increases performance
- Generalizes perfectly to all clients

# 5. Self-Check

- [x] Two research-paper findings were reviewed with constructive methodology questions.
- [x] The source of each finding's label or grouping was identified.
- [x] The limitations of the paper's validation design were explained.
- [x] The Week-5 model was evaluated using both random and grouped client-level splits.
- [x] Before/after validation metrics were compared.
- [x] The grouped split avoided client overlap between training and testing data.
- [x] Leakage-prone trend features were excluded.
- [x] Identifier columns were excluded from model features.
- [x] Missing values were handled through preprocessing.
- [x] Real model limitations and validation gaps were documented.
- [x] Claims were rewritten using evidence-based and decision-support language.
- [x] No guarantees or causal claims were made.